In [1]:
!git clone https://github.com/prava241/flash-attention.git
%cd flash-attention

Cloning into 'flash-attention'...
remote: Enumerating objects: 104, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 104 (delta 49), reused 82 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (104/104), 17.14 KiB | 8.57 MiB/s, done.
Resolving deltas: 100% (49/49), done.
/content/flash-attention


In [4]:
!git pull origin main
!mkdir data
# !git checkout main

From https://github.com/prava241/flash-attention
 * branch            main       -> FETCH_HEAD
Already up to date.


In [5]:
import torch
import numpy as np

N = 4096
D = 256

Q = torch.randn(N, D, device="cuda")
K = torch.randn(N, D, device="cuda")
V = torch.randn(N, D, device="cuda")

Q.cpu().numpy().astype(np.float32).tofile("data/q.bin")
K.cpu().numpy().astype(np.float32).tofile("data/k.bin")
V.cpu().numpy().astype(np.float32).tofile("data/v.bin")

In [12]:
import subprocess

In [9]:
import math

start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)
start.record()

# Naive PyTorch
S = Q @ K.T
S = S / math.sqrt(D)
P = torch.softmax(S, dim=-1)
O = P @ V

#Optimized PyTorch
# O = torch.nn.functional.scaled_dot_product_attention(Q, K, V)

end.record()
torch.cuda.synchronize()
elapsed_ms = start.elapsed_time(end)

print(f"pytorch_attention runtime: {elapsed_ms:.3f} ms")

pytorch_attention runtime: 8.133 ms


In [10]:
# Saves output to data/output.bin, reports runtime
# TODO: Profiles for bottlenecks
!nvcc -O3 src/baseline.cu src/kernels.cu src/utils.cu -o baseline_test

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [38]:
result = subprocess.run(
    ["./baseline_test", "4096", "256"],
    capture_output=True,
    text=True
)

print(result.stdout)

reference = O.cpu().numpy()
output = np.fromfile("data/output.bin", dtype=np.float32).reshape(N, D)

print("Max abs error:", np.max(np.abs(reference - output)))
print("Mean abs error:", np.mean(np.abs(reference - output)))
print("All close:", np.allclose(reference, output, atol=1e-5, rtol=1e-5))

baseline_attention runtime: 92.187 ms

Max abs error: 4.3213367e-07
Mean abs error: 2.1840759e-08
All close: True


In [17]:
!nvcc -O3 src/tiled.cu src/kernels.cu src/utils.cu -o tiled_test

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [48]:
result = subprocess.run(
    ["./tiled_test", "4096", "256"],
    capture_output=True,
    text=True
)

print(result.stdout)

reference = O.cpu().numpy()
output = np.fromfile("data/output.bin", dtype=np.float32).reshape(N, D)

print("Max abs error:", np.max(np.abs(reference - output)))
print("Mean abs error:", np.mean(np.abs(reference - output)))
print("All close:", np.allclose(reference, output, atol=1e-5, rtol=1e-5))

baseline_attention runtime: 70.962 ms

Max abs error: 4.3213367e-07
Mean abs error: 2.1883706e-08
All close: True
